# Acoustic Reef: Model Retraining with Google SurfPerch

This notebook allows you to retrain the Acoustic Reef health classifier using the Google SurfPerch model on Kaggle. 

## Instructions
1. **Add Data**: Upload your audio dataset to Kaggle. You should have a CSV file (e.g., `labels.csv`) mapping filenames to labels.
2. **Enable GPU/Internet**: Ensure Internet access is enabled in Settings to download the SurfPerch model.
3. **Run All**: Execute all cells to train and save the model.

In [ ]:
# Install dependencies
!pip install -q tensorflow_hub librosa scikit-learn seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f"TensorFlow version: {tf.__version__}")

## Configuration
Set the paths to your dataset and output directory.

In [ ]:
# --- CONFIGURATION ---
# Path to the CSV file containing labels
# CSV Format expected: filepath,health_label,anthro_label
LABELS_CSV_PATH = "/kaggle/input/acoustic-reef-data/labels.csv" 

# Root directory for audio files (if paths in CSV are relative)
AUDIO_ROOT_DIR = "/kaggle/input/acoustic-reef-data/audio_files"

# Output directory for the trained model
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ---------------------

## 1. Load SurfPerch Model
We load the pre-trained SurfPerch model from TensorFlow Hub.

In [ ]:
print("Loading SurfPerch model from TFHub...")
model = hub.load("https://tfhub.dev/google/surfperch/1")
print("Model loaded successfully!")

In [ ]:
def load_and_preprocess_audio(file_path, target_sr=32000):
    """Load audio and resample to 32kHz for SurfPerch."""
    try:
        # Load with librosa (auto-resamples if sr specified)
        audio, _ = librosa.load(file_path, sr=target_sr, mono=True)
        
        # Normalize
        max_abs = np.max(np.abs(audio))
        if max_abs > 0:
            audio = audio / max_abs
            
        # Ensure minimum length (1s)
        if len(audio) < target_sr:
            audio = np.pad(audio, (0, target_sr - len(audio)))
            
        return audio
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def get_embedding(audio_data):
    """Generate embedding using SurfPerch."""
    # Prepare input: [1, length]
    audio_tensor = tf.convert_to_tensor(audio_data[np.newaxis, :], dtype=tf.float32)
    
    # Inference
    output = model.signatures['serving_default'](audio_tensor)
    
    # Extract embedding (key 'embedding' or 'output_0')
    if 'embedding' in output:
        emb = output['embedding']
    else:
        emb = next(iter(output.values()))
        
    return emb.numpy()[0]  # Return 1D array

## 2. Process Dataset
Load audio files and generate embeddings.

In [ ]:
if os.path.exists(LABELS_CSV_PATH):
    df = pd.read_csv(LABELS_CSV_PATH)
    print(f"Loaded {len(df)} labels from CSV.")
else:
    print(f"WARNING: Labels CSV not found at {LABELS_CSV_PATH}. Creating dummy data for demonstration.")
    # Dummy data generation if file blocked
    df = pd.DataFrame({
        'filepath': ['dummy_healthy.wav', 'dummy_degraded.wav'],
        'health_label': [1, 0],
        'anthro_label': [0, 1]
    })

embeddings = []
valid_indices = []

print("Generating embeddings...")
for idx, row in df.iterrows():
    # Resolve full path
    fname = row['filepath']
    full_path = os.path.join(AUDIO_ROOT_DIR, fname) if not os.path.isabs(fname) else fname
    
    if not os.path.exists(full_path) and not "dummy" in fname:
        # Try looking in current dir as fallback
        if os.path.exists(fname):
            full_path = fname
        else:
            print(f"File not found: {full_path}")
            continue
            
    if "dummy" in fname:
        # Generate random embedding for demo
        emb = np.random.normal(size=(1280,))
    else:
        audio = load_and_preprocess_audio(full_path)
        if audio is None:
            continue
        emb = get_embedding(audio)
        
    embeddings.append(emb)
    valid_indices.append(idx)

X = np.array(embeddings)
df_clean = df.loc[valid_indices].reset_index(drop=True)
y_health = df_clean['health_label'].values
y_anthro = df_clean['anthro_label'].values

print(f"\nProcessed {len(X)} samples.")
print(f"Embeddings shape: {X.shape}")

## 3. Train Classifier
Train Random Forest classifiers for Health and Anthrophony.

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initialize models
clf_health = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf_anthro = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# Split data
X_train, X_test, yh_train, yh_test, ya_train, ya_test = train_test_split(
    X_scaled, y_health, y_anthro, test_size=0.2, random_state=42
)

print("Training models...")
clf_health.fit(X_train, yh_train)
clf_anthro.fit(X_train, ya_train)
print("Done!")

## 4. Evaluation

In [ ]:
def evaluate_model(clf, X_test, y_test, name):
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"--- {name} Results ---")
    print(f"Accuracy: {acc:.2%}")
    print("\nClassification Report:")
    print(classification_report(y_test, preds))
    
    plt.figure(figsize=(6,4))
    sns.heatmap(confusion_matrix(y_test, preds), annot=True, fmt='d', cmap='Blues')
    plt.title(f'{name} Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

evaluate_model(clf_health, X_test, yh_test, "Reef Health")
evaluate_model(clf_anthro, X_test, ya_test, "Anthrophony")

## 5. Save Model
Save the trained models to be used in the dashboard.

In [ ]:
joblib.dump(clf_health, os.path.join(OUTPUT_DIR, "health_classifier.pkl"))
joblib.dump(clf_anthro, os.path.join(OUTPUT_DIR, "anthro_classifier.pkl"))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.pkl"))

print(f"Models saved to {OUTPUT_DIR}")
print("You can now download these files and place them in your 'models/classifiers' directory locally.")